# Notebook 1: Prepare the 2025 311 Dashboard Dataset

## Purpose
Your original 311 file contains more rows and columns than a Streamlit dashboard needs. In this notebook, you will create a **smaller 2025 deployment dataset**.

You will:
1. Read the full 311 dataset.
2. Convert the request date to a real datetime value.
3. **Keep only requests from 2025.**
4. Create a few useful dashboard variables.
5. Select only the columns your dashboard needs.
6. Save the result as `311_2025_dashboard.csv`.
7. Check the new file size before using GitHub.

> **Important:** You are expected to edit the list of columns. Do not keep a column just because it appears in the starter code.


## 1. Import pandas and locate your file
First, upload the data file downloaded from Blackboard. Then, in Colab, click on the folder icon in the left side menu, then select the arrow in the Files submenu to upload this file. Next, you may need to change the name associated with `FULL_DATA_FILE` so that it matches the name/path of the full 311 file you downloaded. You can see themname under the sample data folder. This is where your completed file will appear, and you will download it from here to upload and commit to your Github repo.


In [5]:
import pandas as pd
from pathlib import Path

# EDIT THIS filename/path if needed
FULL_DATA_FILE = "311_2025_dashboard.csv"

# We will save the smaller file with this name
OUTPUT_FILE = "311_2025_dashboard.csv"


## 2. Read the full dataset
If your source file is a CSV, use the code below. If your instructor supplied a different file type, adjust the read command as directed.


In [6]:
df = pd.read_csv(FULL_DATA_FILE, low_memory=False)

print(f"Rows in full dataset: {len(df):,}")
print(f"Columns in full dataset: {len(df.columns):,}")
df.head()


Rows in full dataset: 518,841
Columns in full dataset: 6


,request_date,requested_datetime,day_of_week,agency_responsible,service_name,zipcode
0,2025-01-01,2025-01-01 00:00:34+00:00,Wednesday,Community Life Improvement Program,Graffiti Removal,19130.0
1,2025-01-01,2025-01-01 00:03:51+00:00,Wednesday,Community Life Improvement Program,Graffiti Removal,19130.0
2,2025-01-01,2025-01-01 00:06:29+00:00,Wednesday,Streets Department,Rubbish/Recyclable Material Collection,19151.0
3,2025-01-01,2025-01-01 00:06:47+00:00,Wednesday,Community Life Improvement Program,Graffiti Removal,19130.0
4,2025-01-01,2025-01-01 00:07:43+00:00,Wednesday,Streets Department,Rubbish/Recyclable Material Collection,19149.0


## 3. Look at the available columns
Before deleting anything, see what is available. Your dashboard should keep only fields that support the questions and charts you want to show.


In [7]:
print(df.columns.tolist())


['request_date', 'requested_datetime', 'day_of_week', 'agency_responsible', 'service_name', 'zipcode']


## 4. Convert the request timestamp and keep ONLY 3 or 4 months in 2025
The dashboard for this project must use **2025 requests only**.

`errors="coerce"` changes invalid date values to missing values instead of crashing the notebook.


In [8]:
df["requested_datetime"] = pd.to_datetime(
    df["requested_datetime"],
    errors="coerce",
    utc=True
)

# Required: keep only the last three months of 2025
df_2025 = df[
    (df["requested_datetime"].dt.year == 2025) &
    (df["requested_datetime"].dt.month >= 10)
].copy()

print(f"Oct-Dec 2025 rows: {len(df_2025):,}")
print("First request:", df_2025["requested_datetime"].min())
print("Last request: ", df_2025["requested_datetime"].max())


Oct-Dec 2025 rows: 122,895
First request: 2025-10-01 00:11:03+00:00
Last request:  2025-12-31 23:51:10+00:00


## 5. Create variables that will be useful in the dashboard
The most important new field is `request_date`. Streamlit will later use it to let the user choose a **start date** and **end date**.

We will also create `month` and `day_of_week`. You may use these in your dashboard, or you may remove them later if you do not need them.


In [9]:
# A date-only field is convenient for dashboard filtering
df_2025["request_date"] = df_2025["requested_datetime"].dt.date

# Optional but useful dashboard variables
df_2025["month"] = df_2025["requested_datetime"].dt.month_name()
df_2025["day_of_week"] = df_2025["requested_datetime"].dt.day_name()

print("Dashboard date range:")
print(df_2025["request_date"].min(), "through", df_2025["request_date"].max())


Dashboard date range:
2025-10-01 through 2025-12-31


## 6. Optional: Calculate resolution time
If you want to analyze how long requests take to close, create `resolution_days`.

If you do **not** plan to use resolution time in your dashboard, you can skip this section.


In [10]:
if "closed_datetime" in df_2025.columns:
    df_2025["closed_datetime"] = pd.to_datetime(
        df_2025["closed_datetime"],
        errors="coerce",
        utc=True
    )

    df_2025["resolution_days"] = (
        df_2025["closed_datetime"] - df_2025["requested_datetime"]
    ).dt.total_seconds() / 86400

    print(df_2025["resolution_days"].describe())


## 7. Choose the columns for YOUR dashboard
The list below is only a starting point. **Edit it.**

Keep fields needed for your charts, filters, metrics, or map. Remove fields you do not use. Large text/media fields such as `media_url`, `status_notes`, `subject`, or `gdb_geomattr_data` usually do not belong in this dashboard dataset unless you have a specific reason to use them.

If you created `resolution_days` and want to use it, add it to the list.


In [16]:
# EDIT THIS LIST based on the dashboard you plan to build
columns_to_keep = [
    "service_request_id",
    "requested_datetime",
    "service_name",
    "agency_responsible",
    "zipcode",
    "day_of_week"
]

# Keep only columns that actually exist in the source file.
# This makes the starter notebook a little more forgiving if column names vary.
columns_to_keep = [c for c in columns_to_keep if c in df_2025.columns]

dashboard_df = df_2025[columns_to_keep].copy()

print(f"Dashboard rows: {len(dashboard_df):,}")
print(f"Dashboard columns: {len(dashboard_df.columns)}")
dashboard_df.head()


Dashboard rows: 122,895
Dashboard columns: 5


,requested_datetime,service_name,agency_responsible,zipcode,day_of_week
395946,2025-10-01 00:11:03+00:00,Other (Streets),Streets Department,19145.0,Wednesday
395947,2025-10-01 00:11:19+00:00,Street Light Outage,Streets Department,19148.0,Wednesday
395948,2025-10-01 00:12:35+00:00,Maintenance Complaint,Community Life Improvement Program,19125.0,Wednesday
395949,2025-10-01 00:14:47+00:00,Sanitation Violation,Streets Department,19104.0,Wednesday
395950,2025-10-01 00:15:23+00:00,Information Request,Philly311 Contact Center,NaN,Wednesday


## 8. Check missing values and data types
Do a quick check before saving. This is not another full EDA; you are simply making sure the deployment data looks reasonable.


In [17]:
print(dashboard_df.dtypes)
print("Missing values:")
print(dashboard_df.isna().sum().sort_values(ascending=False))


requested_datetime    datetime64[ns, UTC]
service_name                       object
agency_responsible                 object
zipcode                           float64
day_of_week                        object
dtype: object
Missing values:
zipcode               71620
requested_datetime        0
service_name              0
agency_responsible        0
day_of_week               0
dtype: int64


## 9. Save the reduced dataset
This is the file that your second notebook and your Streamlit application will use.


In [14]:
dashboard_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")


Saved: 311_2025_dashboard.csv


## 10. Check the saved file size
For this assignment, keep the dashboard data file **under 25 MB** so it is easy to work with in GitHub.

If the file is still too large, go back to your `columns_to_keep` list and remove fields your dashboard does not need. You may also discuss other reasonable reductions with your instructor.


In [15]:
file_size_mb = Path(OUTPUT_FILE).stat().st_size / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

if file_size_mb < 25:
    print("Good: the file is under 25 MB.")
else:
    print("The file is still too large for this assignment. Reduce it further.")


File size: 11.90 MB
Good: the file is under 25 MB.


## 11. Final verification
Your reduced dataset should:
- contain a few months of **2025 only**;
- include `request_date`;
- include only fields that have a purpose in your dashboard;
- be under **25 MB**;
- be saved as `311_2025_dashboard.csv`.

Download the '311_2025_dashboard.csv' file to local storage (click on the dots to the right of the file name under "Files" in your left hand menu). You will then be able to upload this file to your next notebook, and eventually save it to your Github repo that you will create for this project. You are now ready for **Notebook 2: Prototype Your Dashboard Visualizations**.
